# Loading datasets

In [2]:
from datasets import load_dataset
import pandas as pd

judgments = load_dataset("milistu/amazon-esci-data", "queries")
products  = load_dataset("milistu/amazon-esci-data", "products")

esciQueries = pd.concat([judgments["train"].to_pandas(),
                         judgments["test"].to_pandas()], ignore_index=True)
esciProducts = pd.concat([products["train"].to_pandas(),
                         products["test"].to_pandas()], ignore_index=True)

print(esciQueries.shape, esciQueries.columns.tolist())
print(esciProducts.shape, esciProducts.columns.tolist())

(2621288, 10) ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split', '__index_level_0__']
(1814924, 9) ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale', 'split', '__index_level_0__']


In [24]:
esciQueries.query("product_locale == 'us'")

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,__index_level_0__
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,0
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train,1
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train,2
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train,3
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train,4
...,...,...,...,...,...,...,...,...,...,...
2619638,2614589,香奈儿,130378,B00NAGVL7W,us,E,1,1,test,2614589
2619639,2614590,香奈儿,130378,B00HJZT0YQ,us,I,1,1,test,2614590
2619640,2614591,香奈儿,130378,B006IB5T4W,us,I,1,1,test,2614591
2619641,2614592,香奈儿,130378,B0010POWEE,us,I,1,1,test,2614592


In [33]:
esciProducts

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,split,__index_level_0__
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es,train,0
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),None,None,11 Degrees,Blanco,es,train,1
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es,train,2
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,None,None,11 Degrees,None,es,train,3
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,None,None,11 Degrees,Negro (,es,train,4
...,...,...,...,...,...,...,...,...,...
1814919,B08TQT7H55,Yvette（イベット）ノンワイヤーブラジャー ナイトブラ シームレス インナー 完全無縫製...,None,【フリーサイズ】カップAからDまでの全サイズに対応するワンサイズ展開でサイズ選びの悩みを解決...,Yvette,ブラック,jp,test,1814909
1814920,B092V43B6D,【 無縫製ノンワイヤー ブラジャー】Caseeto 女性ブラジャー ラテックスカップ A～D...,商品詳細◆：全体に花柄レースのベースで華やかに胸元を彩る楽チンフルカップブラです。カップのサ...,商品詳細◆：全体に花柄レースのベースで華やかに胸元を彩る楽チンフルカップブラです。カップのサ...,Caseeto,ピンク,jp,test,1814911
1814921,B0971WCFX6,[ウイング/ワコール] ノンワイヤーブラジャー NEW 動いてもズレにくい 【シンクロブラ】...,None,<b>「シンクロブラ」</b><br><b>動きやすい、ズレにくい！</b><br><b>行...,Wing/Wacoal(ウイング/ワコール),OR,jp,test,1814912
1814922,B09B3L32WL,Doyeemei ブラ シームレス紐付き ヌードブラ シリコンブラ ずれにくい盛れるストラッ...,ブラ シームレス紐付き ヌードブラ シリコンブラ ずれにくい盛れるストラップレスブラ ノンワ...,素材構成: 100% 分類外繊維(バイオジェル)。\nサイズ：A・B・C・D（普段より1サイ...,Doyeemei,ブラック,jp,test,1814914


# Reading and parsing local file

In [3]:
import zstandard as zstd
import io
with open("esci.json.zst", "rb") as fh:
    reader = zstd.ZstdDecompressor().stream_reader(fh)
    for line in io.TextIOWrapper(reader, encoding="utf-8"):
        rec = json.loads(line)

KeyboardInterrupt: 

## Parsing functions

In [ ]:
import re
from datetime import datetime

# ---------------------------------------------------------------- stars
_STARS = [
    re.compile(r"([\d.]+)\s+out of\s+5"),        # us: "4.3 out of 5 stars"
    re.compile(r"([\d,]+)\s+de\s+5"),            # es: "4,3 de 5 estrellas"
    re.compile(r"5\s*つ星のうち\s*([\d.]+)"),      # jp: "5つ星のうち4.3"  <- nota vem DEPOIS
]

def parse_stars(s):
    if not s:
        return None
    for rx in _STARS:
        m = rx.search(str(s))
        if m:
            try:
                return float(m.group(1).replace(",", "."))
            except ValueError:
                return None
    return None

# ---------------------------------------------------------------- ratings
# separador de milhar: ',' em us/jp, '.' em es
_RATINGS = [
    (re.compile(r"([\d,]+)\s+(?:global\s+)?ratings?"), ","),
    (re.compile(r"([\d,]+)\s+(?:global\s+)?reviews?"), ","),
    (re.compile(r"([\d.]+)\s+valoraciones?"), "."),
    (re.compile(r"([\d.]+)\s+calificaciones?"), "."),
    (re.compile(r"([\d,]+)\s*個の評価"), ","),
    (re.compile(r"([\d,]+)\s*件のグローバル評価"), ","),
]

def parse_ratings(s):
    if not s:
        return None
    for rx, sep in _RATINGS:
        m = rx.search(str(s))
        if m:
            try:
                return int(m.group(1).replace(sep, ""))
            except ValueError:
                return None
    return None

# ---------------------------------------------------------------- date + country
_MONTHS_ES = {"enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,
              "julio":7,"agosto":8,"septiembre":9,"setiembre":9,"octubre":10,
              "noviembre":11,"diciembre":12}

# o '\n' foi comido pelo scraper e sobrou um 'n' orfao antes da data: "...🇺🇸n September 22, 2022"
_DATE_US = re.compile(   # US: mes-primeiro, com virgula
    r"Reviewed in (?P<country>.+?)\s*[\U0001F1E6-\U0001F1FF]{0,2}\s*n?\s*"
    r"(?:on\s+)?(?P<mon>[A-Z][a-z]+)\s+(?P<day>\d{1,2}),\s*(?P<year>\d{4})"
)
_DATE_UK = re.compile(   # UK/AU: dia-primeiro, sem virgula
    r"Reviewed in (?P<country>.+?)\s*[\U0001F1E6-\U0001F1FF]{0,2}\s*n?\s*"
    r"(?:on\s+)?(?P<day>\d{1,2})\s+(?P<mon>[A-Z][a-z]+)\s+(?P<year>\d{4})"
)
_DATE_ES = re.compile(
    r"(?:Revisado|Comentado) en (?P<country>.+?)\s*[\U0001F1E6-\U0001F1FF]{0,2}\s*n?\s*"
    r"el\s+(?P<day>\d{1,2})\s+de\s+(?P<mon>\w+)\s+de\s+(?P<year>\d{4})"
)
_DATE_JP = re.compile(
    r"(?P<year>\d{4})年\s*(?P<mon>\d{1,2})月\s*(?P<day>\d{1,2})日に(?P<country>.+?)でレビュー済み"
)

def parse_review_date(s):
    """-> (country, datetime | None). (None, None) se nao parsear."""
    if not s:
        return None, None
    s = str(s)

    for rx in (_DATE_US, _DATE_UK):
        m = rx.search(s)
        if m:
            try:
                dt = datetime.strptime(f"{m['mon']} {m['day']} {m['year']}", "%B %d %Y")
            except ValueError:
                dt = None
            return m["country"].strip(), dt

    m = _DATE_ES.search(s)
    if m:
        mon = _MONTHS_ES.get(m["mon"].lower())
        dt = datetime(int(m["year"]), mon, int(m["day"])) if mon else None
        return m["country"].strip(), dt

    m = _DATE_JP.search(s)
    if m:
        try:
            dt = datetime(int(m["year"]), int(m["mon"]), int(m["day"]))
        except ValueError:
            dt = None
        return m["country"].strip(), dt

    return None, None

## Write as parquet files

In [ ]:
import io, json
import zstandard as zstd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

def iter_records(path):
    with open(path, "rb") as fh:
        reader = zstd.ZstdDecompressor().stream_reader(fh)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            line = line.strip()
            if line:
                yield json.loads(line)

# Schema explicito: o ParquetWriter exige consistencia entre chunks, e
# inferencia por chunk vai divergir em dado heterogeneo.
prod_schema = pa.schema([
    ("asin", pa.string()), ("type", pa.string()), ("locale", pa.string()),
    ("template", pa.string()), ("title", pa.string()),
    ("stars", pa.float32()), ("n_ratings", pa.int32()),
    ("n_reviews", pa.int32()), ("cat_top", pa.string()),
    ("category", pa.list_(pa.string())), ("price", pa.string()),
    ("image", pa.string()), ("n_attrs", pa.int32()),
    ("description", pa.string()),
])

rev_schema = pa.schema([
    ("asin", pa.string()), ("stars", pa.float32()), ("title", pa.string()),
    ("text", pa.string()), ("text_len", pa.int32()),
    ("country", pa.string()), ("date", pa.timestamp("s")),
])

def flush(writer, rows, schema):
    if rows:
        writer.write_table(pa.Table.from_pylist(rows, schema=schema))
    return []

CHUNK = 100_000
prod_rows, rev_rows = [], []

with pq.ParquetWriter("esci_s_products.parquet", prod_schema, compression="zstd") as pw, \
     pq.ParquetWriter("esci_s_reviews.parquet", rev_schema, compression="zstd") as rw:

    for r in tqdm(iter_records("esci.json.zst"), total=1_661_908):
        asin = r.get("asin")
        revs = r.get("reviews") or []
        cat = r.get("category") or []

        prod_rows.append({
            "asin": asin, "type": r.get("type"), "locale": r.get("locale"),
            "template": r.get("template"), "title": r.get("title"),
            "stars": parse_stars(r.get("stars")),
            "n_ratings": parse_ratings(r.get("ratings")),
            "n_reviews": len(revs),
            "cat_top": cat[0] if cat else None,
            "category": cat,
            "price": r.get("price"), "image": r.get("image"),
            "n_attrs": len(r.get("attrs") or {}),
            "description": r.get("description"),
        })

        for rev in revs:
            country, date = parse_review_date(rev.get("date"))
            txt = rev.get("text") or ""
            rev_rows.append({
                "asin": asin, "stars": parse_stars(rev.get("stars")),
                "title": rev.get("title"), "text": txt, "text_len": len(txt),
                "country": country, "date": date,
            })

        if len(prod_rows) >= CHUNK:
            prod_rows = flush(pw, prod_rows, prod_schema)
        if len(rev_rows) >= CHUNK:
            rev_rows = flush(rw, rev_rows, rev_schema)

    flush(pw, prod_rows, prod_schema)
    flush(rw, rev_rows, rev_schema)

# Reading ESCI Extended parquet files

In [24]:
# esciExtendedProducts = pd.read_parquet("esci_s_products.parquet", dtype_backend="pyarrow")
# print(esciExtendedProducts.shape)
# esciExtendedProducts.head()

esciExtendedProducts = pd.read_parquet("esci_s_products.parquet",
                     columns=["asin", "type", "title", "locale", "n_reviews"],
                     dtype_backend="pyarrow")

In [5]:
esciExtendedReviews = pd.read_parquet("esci_s_reviews.parquet", dtype_backend="pyarrow")
print(esciExtendedReviews.shape)
esciExtendedReviews.head()

(13050444, 7)


,asin,stars,title,text,text_len,country,date
0,B06XXK1JQN,4.0,ESTORES FANTASTICOS,La calidad no está mal y no tiene nada que env...,228,España,2018-02-17 00:00:00
1,B06XXK1JQN,4.0,"Calidad precio si no se usa mucho , bueno",Precio bien Montaje bien Los bordes se deshiel...,140,España,2018-09-18 00:00:00
2,B06XXK1JQN,3.0,Cumple función básica,"Calidad no es muy buena, para un apaño, sencil...",59,España,2019-09-21 00:00:00
3,B06XXK1JQN,4.0,Se deshilacha,Se deshilacha,13,España,2019-08-25 00:00:00
4,B06XXK1JQN,3.0,Buen precio,El material es un poco tipo papel... Pero por ...,109,España,2019-01-03 00:00:00


# Filter & join

In [27]:
locations = ['us']

print(esciProducts.query("product_locale in @locations").product_id.nunique())
print(esciExtendedProducts.query("locale in @locations").asin.nunique())

1215854
1118658


In [29]:
esciProductsUS = pd.merge(esciProducts.query("product_locale in @locations"), esciExtendedProducts.query("locale in @locations"),
              left_on="product_id", right_on="asin",
              how="inner")

print(esciProductsUS.product_id.nunique())

1118658


In [34]:
esciProductsUS

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,split,__index_level_0__,asin,type,title,locale,n_reviews
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us,train,167168,B003O0MNGC,product,Delta Electronics (Americas) Ltd. VFB25ACH Exh...,us,8
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us,train,167169,B00MARNO5Y,product,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,us,9
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us,train,167170,B011RX6PNO,product,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,us,8
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us,train,167171,B01MZIK0PI,product,Delta Electronics (Americas) Ltd. RAD80 Delta ...,us,6
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us,train,167172,B01N5Y6002,product,Delta Electronics (Americas) Ltd. GBR80HLED De...,us,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118653,B08XX3NL14,"SAYFINE 5 Pack Plain Atheletic Shirts for Men,...",<b>SAYFINE MENS T SHIRTS & GYM T SHIRTS & WORK...,[VALUE PACK] - Athletic crew neck and flat-loc...,SAYFINE,Black (5 Pack),us,test,1753347,B08XX3NL14,product,"SAYFINE 5 Pack Plain Atheletic Shirts for Men,...",us,9
1118654,B08Y8M9Z6L,GEEK LIGHTING Mens Polo Shirt Sport Casual Sho...,None,Material: This mens running shirt is made with...,GEEK LIGHTING,012-grey,us,test,1753348,B08Y8M9Z6L,product,GEEK LIGHTING Mens Polo Shirt Sport Casual Sho...,us,5
1118655,B0936FKSS6,Alex Vando Mens Golf Shirt Moisture Wicking Qu...,None,Material: Our Mens Golf Shirts are made from 9...,Alex Vando,Blue,us,test,1753349,B0936FKSS6,product,Alex Vando Mens Golf Shirt Moisture Wicking Qu...,us,8
1118656,B096RXY1VT,Casei Womens Polo Shirts Golf Shirts Quick Dry...,black friday deals 2021 gifts for men gifts fo...,【Quick Dry & Moisture Wicking】Womens polo shir...,Casei,White,us,test,1753350,B096RXY1VT,product,Casei Womens Polo Shirts Golf Shirts Quick Dry...,us,8


In [31]:
t1_asins = set(esciQueries.query("small_version == 1 and product_locale == 'us'").product_id)

sub = esciProductsUS[esciProductsUS.product_id.isin(t1_asins)]
com = (sub.n_reviews > 0).sum()
print(f"task1/us: {len(t1_asins):,} ASINs")
print(f"  com metadata: {len(sub):,} ({100*len(sub)/len(t1_asins):.1f}%)")
print(f"  com review:   {com:,} ({100*com/len(t1_asins):.1f}%)")

task1/us: 482,105 ASINs
  com metadata: 447,924 (92.9%)
  com review:   412,693 (85.6%)


In [35]:
com = (esciProductsUS.n_reviews > 0).sum()
tot = len(esciProductsUS)
print(f"com >=1 review: {com:,} / {tot:,}  ({100*com/tot:.1f}%)")
print(esciProductsUS.n_reviews.describe(percentiles=[.1,.25,.5,.75,.9]))

com >=1 review: 1,023,287 / 1,118,658  (91.5%)
count    1118658.0
mean      8.526826
std       4.029636
min            0.0
10%            1.0
25%            8.0
50%            8.0
75%           13.0
90%           13.0
max           13.0
Name: n_reviews, dtype: double[pyarrow]


# Generating final datasets

In [38]:
locations = ['us']

# ---------------------------------------------------------------- 0. recarregar completo
# a celula 11 carregou so 5 colunas; o parquet tem 14
esciExtendedProducts = pd.read_parquet(
    "esci_s_products.parquet",
    columns=["asin", "locale", "type", "template", "category", "cat_top",
             "stars", "n_ratings", "n_reviews", "price", "image", "n_attrs"],
    dtype_backend="pyarrow",
)   # 'title' e 'description' ficam de fora: o ESCI ja tem os dele, e sao
    # esses dois que estouram o offset overflow do Arrow no merge

t1_asins = set(
    esciQueries.query("small_version == 1 and product_locale in @locations").product_id
)
print(f"task1/us: {len(t1_asins):,} ASINs")

task1/us: 482,105 ASINs


In [39]:
# ---------------------------------------------------------------- 1. PRODUCTS
products = (
    pd.merge(
        esciProducts.query("product_locale in @locations")[
            ["product_id", "product_title", "product_description",
             "product_bullet_point", "product_brand", "product_color"]
        ],
        esciExtendedProducts.query("locale in @locations").drop(columns=["locale"]),
        left_on="product_id", right_on="asin", how="inner",
        validate="one_to_one",          # levanta erro se houver fan-out
    )
    .drop(columns=["asin"])
    .query("product_id in @t1_asins")
    .reset_index(drop=True)
)

print(f"products: {products.shape}")
print(f"  ASINs unicos: {products.product_id.nunique():,}  (len={len(products):,})")
print(f"  com review:   {(products.n_reviews > 0).sum():,}")
products.head(3)

products: (447924, 16)
  ASINs unicos: 447,924  (len=447,924)
  com review:   412,693


,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,type,template,category,cat_top,stars,n_ratings,n_reviews,price,image,n_attrs
0,B0188A3QRM,"Amazon Basics Woodcased #2 Pencils, Unsharpene...",None,144 woodcase #2 HB pencils made from high-qual...,Amazon Basics,Yellow,product,office_product,['Office Products' 'Office & School Supplies' ...,Office Products,4.8,78264,13,,https://m.media-amazon.com/images/I/71ktFdNm+x...,5
1,B075VXJ9VG,"BAZIC Pencil #2 HB Pencils, Latex Free Eraser,...",<p><strong>BACK TO BAZIC</strong></p><p>Our go...,&#11088; UN-SHARPENED #2 PREMIUM PENCILS. Each...,BAZIC Products,12-count,product,office_product,['Office Products' 'Office & School Supplies' ...,Office Products,4.6,22,1,,https://m.media-amazon.com/images/I/710GRlD5pi...,5
2,B07G7F6JZ6,Emraw Pre Sharpened Round Primary Size No 2 Ju...,<p><b>Emraw Pre-Sharpened #2 HB Wood Pencils -...,✓ PACK OF 8 NUMBER 2 PRESHARPENED BEGINNERS PE...,Emraw,Yellow,product,office_product,['Office Products' 'Office & School Supplies' ...,Office Products,4.8,1071,8,,https://m.media-amazon.com/images/I/71Otxi4tUH...,5


In [40]:
# ---------------------------------------------------------------- 2. QRELS
# Grao: (query_id, product_id). Isto e o qrels do TREC.
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

in_scope = set(products.product_id)

qrels = (
    esciQueries
    .query("small_version == 1 and product_locale in @locations")
    [["query_id", "query", "product_id", "esci_label", "split"]]
    .query("product_id in @in_scope")
    .assign(gain=lambda d: d.esci_label.map(GAIN))
    .reset_index(drop=True)
)

print(f"qrels: {qrels.shape}  |  {qrels.query_id.nunique():,} queries")
print(qrels.esci_label.value_counts(normalize=True).mul(100).round(1))
print(qrels.split.value_counts())

qrels: (561834, 6)  |  29,833 queries
esci_label
E    43.2
S    35.3
I    17.0
C     4.5
Name: proportion, dtype: float64
split
train    391955
test     169879
Name: count, dtype: int64


In [43]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("esci_s_reviews.parquet")
chunks = []
for batch in pf.iter_batches(batch_size=500_000):
    df = batch.to_pandas()                      # batch pequeno, nao estoura
    chunks.append(df[df.asin.isin(in_scope)])
reviews = pd.concat(chunks, ignore_index=True).rename(
    columns={"stars": "rev_stars", "title": "rev_title"}
)
print(reviews.shape)
reviews.to_parquet("task1_us_reviews.parquet", compression="zstd")

(3962238, 7)


In [44]:
products.to_parquet("task1_us_products.parquet", compression="zstd")
qrels.to_parquet("task1_us_qrels.parquet", compression="zstd")
reviews.to_parquet("task1_us_reviews.parquet", compression="zstd")